In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset.csv to dataset (2).csv
Saving symptom_Description.csv to symptom_Description (1).csv
Saving symptom_precaution.csv to symptom_precaution (1).csv


In [34]:
import pandas as pd
import numpy as np

df = pd.read_csv("dataset.csv")
desc_df = pd.read_csv("symptom_Description.csv")
prec_df = pd.read_csv("symptom_precaution.csv")

In [35]:
desc_df.columns = desc_df.columns.str.strip()
prec_df.columns = prec_df.columns.str.strip()

In [36]:
disease_desc = dict(zip(desc_df["Disease"], desc_df["Description"]))

disease_prec = {}

for i in range(len(prec_df)):
    disease = prec_df.iloc[i]["Disease"]
    precautions = prec_df.iloc[i][1:].dropna().values.tolist()
    disease_prec[disease] = precautions

In [37]:
y = df["Disease"]
X = df.drop("Disease", axis=1)

In [43]:
df.fillna("None", inplace=True)

In [47]:
import numpy as np

# Convert ALL values to string (VERY IMPORTANT FIX)
X = X.astype(str)

# Now safely get unique symptoms
symptoms = np.unique(X.values.flatten())

# Remove 'None'
symptoms = [s for s in symptoms if s != "None"]

# Create encoded dataframe
X_encoded = pd.DataFrame(0, index=df.index, columns=symptoms)

# Fill values
for col in X.columns:
    for i in range(len(df)):
        symptom = X.iloc[i][col]
        if symptom != "None":
            X_encoded.loc[i, symptom] = 1

# Replace X
X = X_encoded

In [48]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [49]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [50]:
from sklearn.ensemble import RandomForestClassifier

final_model = RandomForestClassifier()
final_model.fit(X_train, y_train)

RandomForestClassifier()

In [51]:
from sklearn.metrics import accuracy_score

pred = final_model.predict(X_test)
acc = accuracy_score(y_test, pred)

print("Model Accuracy:", acc)

Model Accuracy: 1.0


In [52]:
sample = X.iloc[[0]]

prediction = final_model.predict(sample)
disease = le.inverse_transform(prediction)[0]

print("Predicted Disease:", disease)

Predicted Disease: Fungal infection


In [53]:
description = disease_desc.get(disease, "No description available")
precautions = disease_prec.get(disease, ["No precautions available"])

In [55]:
print("Predicted Disease:", disease)
print("\nDescription:")
print(description)

print("\nPrecautions:")
for i, p in enumerate(precautions, 1):
    print(f"{i}. {p}")

Predicted Disease: Fungal infection

Description:
In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune system to handle. Fungi can live in the air, soil, water, and plants. There are also some fungi that live naturally in the human body. Like many microbes, there are helpful fungi and harmful fungi.

Precautions:
1. bath twice
2. use detol or neem in bathing water
3. keep infected area dry
4. use clean cloths


In [56]:
import pickle

pickle.dump(final_model, open("model.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))